In [1]:
from pathlib import Path
import copy, json, os, random, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Confirming CUDA is being used and Reproductibility

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "vit_b16_hparam"
REPORT_DIR = PROJECT_ROOT / "reports" / "vit_b16_hparam"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PyTorch: 2.14.0+cu130
CUDA runtime: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 2. Exact baseline split and image paths

In [2]:
df = pd.read_csv(
    DATA_DIR / "interim" / "compcars_resnet50_baseline_split.csv"
)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

NUM_CLASSES = df["target"].nunique()
class_names = (
    df[["target", "class_name"]]
    .drop_duplicates()
    .sort_values("target")["class_name"]
    .tolist()
)

DEFAULT_COMPCARS = r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\compcars_kaggle\extracted"
LOCAL_COMPCARS_EXTRACTED = Path(
    os.environ.get("COMPCARS_EXTRACTED", DEFAULT_COMPCARS)
)

def resolve_local_path(original_path):
    text = str(original_path).replace("\\", "/")
    if "/extracted/" in text:
        relative = text.split("/extracted/", 1)[1]
    elif "/image/" in text:
        relative = "image/" + text.split("/image/", 1)[1]
    else:
        return Path(original_path)
    return LOCAL_COMPCARS_EXTRACTED / Path(*relative.split("/"))

for part in [train_df, val_df, test_df]:
    part["local_file_path"] = part["file_path"].map(resolve_local_path)

print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Classes:", NUM_CLASSES)

Train / Val / Test: 1600 200 200
Classes: 20


## 3. Experiment configurations

In [3]:
EXPERIMENTS = [
    {
        "name": "baseline",
        "finetune_lr": 5e-5,
        "weight_decay": 1e-4,
        "last_blocks": 1,
        "label_smoothing": 0.1,
        "augmentation": "standard",
    },
    {
        "name": "lr_1e-5",
        "finetune_lr": 1e-5,
        "weight_decay": 1e-4,
        "last_blocks": 1,
        "label_smoothing": 0.1,
        "augmentation": "standard",
    },
    {
        "name": "lr_1e-4",
        "finetune_lr": 1e-4,
        "weight_decay": 1e-4,
        "last_blocks": 1,
        "label_smoothing": 0.1,
        "augmentation": "standard",
    },
    {
        "name": "unfreeze_2_blocks",
        "finetune_lr": 5e-5,
        "weight_decay": 1e-4,
        "last_blocks": 2,
        "label_smoothing": 0.1,
        "augmentation": "standard",
    },
    {
        "name": "unfreeze_4_blocks",
        "finetune_lr": 2e-5,
        "weight_decay": 1e-4,
        "last_blocks": 4,
        "label_smoothing": 0.1,
        "augmentation": "standard",
    },
    {
        "name": "no_label_smoothing",
        "finetune_lr": 5e-5,
        "weight_decay": 1e-4,
        "last_blocks": 1,
        "label_smoothing": 0.0,
        "augmentation": "standard",
    },
    {
        "name": "strong_augmentation",
        "finetune_lr": 5e-5,
        "weight_decay": 1e-4,
        "last_blocks": 1,
        "label_smoothing": 0.1,
        "augmentation": "strong",
    },
    {
        "name": "weight_decay_1e-5",
        "finetune_lr": 5e-5,
        "weight_decay": 1e-5,
        "last_blocks": 1,
        "label_smoothing": 0.1,
        "augmentation": "standard",
    },
]

display(pd.DataFrame(EXPERIMENTS))

,name,finetune_lr,weight_decay,last_blocks,label_smoothing,augmentation
0,baseline,0.00005,0.00010,1,0.1,standard
1,lr_1e-5,0.00001,0.00010,1,0.1,standard
2,lr_1e-4,0.00010,0.00010,1,0.1,standard
3,unfreeze_2_blocks,0.00005,0.00010,2,0.1,standard
4,unfreeze_4_blocks,0.00002,0.00010,4,0.1,standard
5,no_label_smoothing,0.00005,0.00010,1,0.0,standard
6,strong_augmentation,0.00005,0.00010,1,0.1,strong
7,weight_decay_1e-5,0.00005,0.00001,1,0.1,standard


## 4. Transforms and dataset

In [4]:
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def make_train_transform(kind):
    if kind == "standard":
        return transforms.Compose([
            transforms.RandomResizedCrop(
                IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.85, 1.15)
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=0.15, contrast=0.15, saturation=0.10
            ),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])

    if kind == "strong":
        return transforms.Compose([
            transforms.RandomResizedCrop(
                IMAGE_SIZE, scale=(0.70, 1.0), ratio=(0.80, 1.20)
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(5),
            transforms.ColorJitter(
                brightness=0.25, contrast=0.25,
                saturation=0.20, hue=0.03
            ),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
            transforms.RandomErasing(p=0.20, scale=(0.02, 0.10)),
        ])

    raise ValueError(kind)

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CompCarsDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["local_file_path"]).convert("RGB")
        return self.transform(image), int(row["target"])

BATCH_SIZE = 8
NUM_WORKERS = 0

def make_loaders(augmentation):
    train_loader = DataLoader(
        CompCarsDataset(train_df, make_train_transform(augmentation)),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    val_loader = DataLoader(
        CompCarsDataset(val_df, eval_transform),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        CompCarsDataset(test_df, eval_transform),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    return train_loader, val_loader, test_loader

## 5. Training utilities

In [5]:
def build_vit():
    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)

    for p in model.parameters():
        p.requires_grad = False

    model.heads.head = nn.Linear(
        model.heads.head.in_features,
        NUM_CLASSES,
    )

    return model.to(DEVICE)

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    correct = 0
    samples = 0

    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits = model(images)
            loss = criterion(logits, targets)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = images.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(1) == targets).sum().item()
        samples += bs

    return total_loss / samples, correct / samples

def train_epochs(model, train_loader, val_loader, criterion, optimizer, epochs):
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    best_val = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()

        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, scaler
        )

        with torch.inference_mode():
            val_loss, val_acc = run_epoch(
                model, val_loader, criterion
            )

        elapsed = time.perf_counter() - start

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_top1": train_acc,
            "val_loss": val_loss,
            "val_top1": val_acc,
            "seconds": elapsed,
        })

        print(
            f"epoch {epoch}/{epochs} | "
            f"train={train_acc:.3f} val={val_acc:.3f} | "
            f"{elapsed:.1f}s"
        )

        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history, best_val, best_state

## 6. Running the hyperparameter study

In [6]:
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 3
HEAD_LR = 1e-3

search_results = []

for config in EXPERIMENTS:
    print("\n" + "=" * 80)
    print("EXPERIMENT:", config["name"])
    print(config)
    print("=" * 80)

    train_loader, val_loader, _ = make_loaders(
        config["augmentation"]
    )

    model = build_vit()

    criterion = nn.CrossEntropyLoss(
        label_smoothing=config["label_smoothing"]
    )

    # Stage 1: classifier head
    optimizer = torch.optim.AdamW(
        model.heads.head.parameters(),
        lr=HEAD_LR,
        weight_decay=config["weight_decay"],
    )

    hist1, val1, state1 = train_epochs(
        model, train_loader, val_loader,
        criterion, optimizer, STAGE1_EPOCHS
    )

    # Stage 2: configurable number of final transformer blocks
    for p in model.encoder.layers[-config["last_blocks"]:].parameters():
        p.requires_grad = True
    for p in model.encoder.ln.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["finetune_lr"],
        weight_decay=config["weight_decay"],
    )

    hist2, val2, state2 = train_epochs(
        model, train_loader, val_loader,
        criterion, optimizer, STAGE2_EPOCHS
    )

    if val1 > val2:
        selected_stage = "head"
        selected_val = val1
        selected_state = state1
    else:
        selected_stage = "fine_tune"
        selected_val = val2
        selected_state = state2

    checkpoint_path = MODEL_DIR / f"{config['name']}_best.pt"
    torch.save({
        "state_dict": selected_state,
        "config": config,
        "validation_top1": float(selected_val),
        "selected_stage": selected_stage,
        "classes": class_names,
    }, checkpoint_path)

    search_results.append({
        **config,
        "selected_stage": selected_stage,
        "validation_top1": selected_val,
        "checkpoint": str(checkpoint_path),
        "total_seconds": sum(
            row["seconds"] for row in hist1 + hist2
        ),
    })

    del model
    torch.cuda.empty_cache()

search_df = pd.DataFrame(search_results).sort_values(
    "validation_top1", ascending=False
).reset_index(drop=True)

display(search_df)
search_df.to_csv(
    REPORT_DIR / "hyperparameter_search.csv",
    index=False,
)


EXPERIMENT: baseline
{'name': 'baseline', 'finetune_lr': 5e-05, 'weight_decay': 0.0001, 'last_blocks': 1, 'label_smoothing': 0.1, 'augmentation': 'standard'}


epoch 1/3 | train=0.289 val=0.440 | 70.5s


epoch 2/3 | train=0.595 val=0.525 | 53.2s


epoch 3/3 | train=0.716 val=0.645 | 52.8s


epoch 1/3 | train=0.840 val=0.665 | 51.5s


...

epoch 3/3 | train=0.721 val=0.625 | 53.2s


epoch 1/3 | train=0.838 val=0.650 | 55.0s


epoch 2/3 | train=0.914 val=0.690 | 55.8s


epoch 3/3 | train=0.967 val=0.720 | 55.4s


,name,finetune_lr,weight_decay,last_blocks,label_smoothing,augmentation,selected_stage,validation_top1,checkpoint,total_seconds
0,unfreeze_4_blocks,0.00002,0.00010,4,0.1,standard,fine_tune,0.785,C:\Users\elesh\Academics\UTS\36127InnovationLa...,333.354709
1,unfreeze_2_blocks,0.00005,0.00010,2,0.1,standard,fine_tune,0.775,C:\Users\elesh\Academics\UTS\36127InnovationLa...,320.832622
2,lr_1e-4,0.00010,0.00010,1,0.1,standard,fine_tune,0.745,C:\Users\elesh\Academics\UTS\36127InnovationLa...,347.355618
3,strong_augmentation,0.00005,0.00010,1,0.1,strong,fine_tune,0.730,C:\Users\elesh\Academics\UTS\36127InnovationLa...,358.444107
4,no_label_smoothing,0.00005,0.00010,1,0.0,standard,fine_tune,0.720,C:\Users\elesh\Academics\UTS\36127InnovationLa...,332.013521
5,weight_decay_1e-5,0.00005,0.00001,1,0.1,standard,fine_tune,0.720,C:\Users\elesh\Academics\UTS\36127InnovationLa...,326.291533
6,baseline,0.00005,0.00010,1,0.1,standard,fine_tune,0.705,C:\Users\elesh\Academics\UTS\36127InnovationLa...,338.656520
7,lr_1e-5,0.00001,0.00010,1,0.1,standard,fine_tune,0.660,C:\Users\elesh\Academics\UTS\36127InnovationLa...,341.248754


## 7. Selecting the best configuration

In [7]:
best = search_df.iloc[0]

print("BEST CONFIGURATION")
display(best.to_frame().T)

BEST CONFIGURATION


,name,finetune_lr,weight_decay,last_blocks,label_smoothing,augmentation,selected_stage,validation_top1,checkpoint,total_seconds
0,unfreeze_4_blocks,0.00002,0.0001,4,0.1,standard,fine_tune,0.785,C:\Users\elesh\Academics\UTS\36127InnovationLa...,333.354709


## 8. Running a final test on the best configuration

In [9]:
best_config = {
    key: best[key]
    for key in [
        "name", "finetune_lr", "weight_decay",
        "last_blocks", "label_smoothing", "augmentation"
    ]
}

_, _, test_loader = make_loaders(best_config["augmentation"])

model = build_vit()

checkpoint = torch.load(
    best["checkpoint"],
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

y_true = []
y_pred = []
all_probs = []

with torch.inference_mode():
    for images, targets in tqdm(test_loader):
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)

        y_true.extend(targets.numpy())
        y_pred.extend(probs.argmax(1).cpu().numpy())
        all_probs.append(probs.cpu())

y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)
all_probs = torch.cat(all_probs)

top1 = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

top5_idx = all_probs.topk(5, dim=1).indices.numpy()
top5 = np.mean([
    true in candidates
    for true, candidates in zip(y_true, top5_idx)
])

print(f"Best config test Top-1: {top1:.4f}")
print(f"Best config test Top-5: {top5:.4f}")
print(f"Best config Macro F1:   {macro_f1:.4f}")

report = pd.DataFrame(
    classification_report(
        y_true, y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).T

report.to_csv(REPORT_DIR / "best_config_classification_report.csv")

final_metrics = {
    "best_config": best_config,
    "validation_top1": float(best["validation_top1"]),
    "test_top1": float(top1),
    "test_top5": float(top5),
    "test_macro_f1": float(macro_f1),
    "gpu": GPU_NAME,
}

def json_converter(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


with (REPORT_DIR / "best_config_metrics.json").open("w") as f:
    json.dump(
        final_metrics,
        f,
        indent=2,
        default=json_converter,
    )

print("Saved:", REPORT_DIR / "best_config_metrics.json")
final_metrics

final_metrics

100%|██████████| 25/25 [00:05<00:00,  4.58it/s]

Best config test Top-1: 0.7800
Best config test Top-5: 0.9950
Best config Macro F1:   0.7760
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\vit_b16_hparam\best_config_metrics.json


{'best_config': {'name': 'unfreeze_4_blocks',
  'finetune_lr': np.float64(2e-05),
  'weight_decay': np.float64(0.0001),
  'last_blocks': np.int64(4),
  'label_smoothing': np.float64(0.1),
  'augmentation': 'standard'},
 'validation_top1': 0.785,
 'test_top1': 0.78,
 'test_top5': 0.995,
 'test_macro_f1': 0.776034842780199,
 'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU'}